# GAN-RL Protein Function Prediction
**Goal:** Beat ProtHGT-ESM2 Biological Process Fmax (baseline: 0.7489)

**Workflow:** Cell 1 → Cell 11 in order. Checkpoints are saved to Drive after each phase.

In [1]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [2]:
# ── Cell 2: Install Dependencies ───────────────────────────────────────────────
# Colab already has PyTorch — do NOT reinstall it.
# We detect the pre-installed torch/CUDA version and pull matching PyG wheels.
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]             # e.g. '2.3.0'
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') # e.g. 'cu121'
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
print(f'Detected: torch={torch_ver}, cuda={cuda_tag}')
print(f'PyG wheel URL: {pyg_url}')

print('Installing torch_geometric ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)

print('Installing PyG sparse extensions ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', pyg_url, '-q'
], check=True)

print('Installing other deps ...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'networkx', 'pyyaml', 'obonet',
    'scikit-learn', 'tqdm', 'pandas',
    'matplotlib', 'tensorboard', '-q'], check=True)

print('Done. No runtime restart needed.')

Detected: torch=2.11.0, cuda=cu128
PyG wheel URL: https://data.pyg.org/whl/torch-2.11.0+cu128.html
Installing torch_geometric ...
Installing PyG sparse extensions ...
Installing other deps ...
Done. No runtime restart needed.


In [3]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT     = '/content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/'
CHECKPOINT_DIR = '/content/drive/MyDrive/Poster code/checkpoints/'
LOG_CSV        = os.path.join(CHECKPOINT_DIR, 'training_log.csv')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Sanity check — verify ESM2 split files are in place
esm2_dir = os.path.join(DRIVE_ROOT, 'alternative_protein_embeddings/esm2/')
expected = [
    'prothgt-esm2-train-graph.pt',
    'prothgt-esm2-val-graph.pt',
    'prothgt-esm2-test-graph.pt',   # optional — val used as proxy if missing
]
for f in expected:
    path = os.path.join(esm2_dir, f)
    exists = os.path.exists(path)
    tag = 'OK' if exists else ('MISSING (optional)' if 'test' in f else 'MISSING')
    print(f'  {f}: {tag}')

print(f'\nDRIVE_ROOT:     {DRIVE_ROOT}')
print(f'CHECKPOINT_DIR: {CHECKPOINT_DIR}')

Mounted at /content/drive
  prothgt-esm2-train-graph.pt: OK
  prothgt-esm2-val-graph.pt: OK
  prothgt-esm2-test-graph.pt: OK

DRIVE_ROOT:     /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/
CHECKPOINT_DIR: /content/drive/MyDrive/Poster code/checkpoints/


In [4]:
# ── Cell 4: Clone Repository ───────────────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/Prot_B_poster'
REPO_URL = 'https://github.com/Drjay806/Prot_B_poster.git'

if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, 'src')):
    print('Repo already present — pulling latest ...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print('Cloning repo ...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR cloning repo:')
        print(result.stderr)
        raise RuntimeError('Git clone failed — push your code first.')
    print(result.stdout)

# Install src as an editable package — fixes "No module named src" permanently.
# pip install -e registers src/ in Python's site-packages so any cell can import it
# without sys.path hacks, even after Drive remounts or kernel restarts within the session.
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR, '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip install -e failed:', result.stderr)
else:
    print('src package installed.')

print(f'Repo ready at {REPO_DIR}')
print('Contents:', os.listdir(REPO_DIR))

Cloning repo ...

src package installed.
Repo ready at /content/Prot_B_poster
Contents: ['prot_b_poster.egg-info', '.git', 'src', 'configs', 'notebooks', 'scripts', 'setup.py', 'requirements.txt']


In [5]:
# ── Cell 5: Load Config ────────────────────────────────────────────────────────
import yaml, os

config_path = os.path.join(REPO_DIR, 'configs/default.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Override paths with what we set above
cfg['data']['drive_root'] = DRIVE_ROOT
cfg['data']['checkpoint_dir'] = CHECKPOINT_DIR

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Target ontology:', cfg['data']['ontology'])
print('Target node type:', cfg['data']['target_type'])

Device: cuda
Target ontology: bp
Target node type: GO_term_P


In [6]:
import sys
sys.path.insert(0, REPO_DIR)

# ── Cell 6: Load Data + Build Hierarchy Tables ────────────────────────────────
# Data is loaded onto CPU. CompGCN moves only the relevant ~150 MB of node
# features to GPU per forward pass, freeing ~3 GB of VRAM for activations.

from src.data.loader import load_prothgt_splits
from src.data.go_hierarchy import build_ancestor_table, build_propagation_edges

splits = load_prothgt_splits(
    drive_root=cfg['data']['drive_root'],
    ontology=cfg['data']['ontology'],
    device='cpu',   # keep on CPU — encoder handles GPU placement internally
)
train_data, val_data, test_data = splits.train, splits.val, splits.test
target_type = splits.target_type

print(f'\nNode types: {train_data.node_types}')
print(f'Proteins:   {train_data["Protein"].x.shape[0]:,}')
print(f'GO terms:   {train_data[target_type].x.shape[0]:,}')

print('\nBuilding GO ancestor table (used by RL reward) ...')
ancestor_table = build_ancestor_table(train_data, target_type=target_type, cache=True)

print('Building propagation edge list (used by evaluation) ...')
prop_edges = build_propagation_edges(train_data, target_type=target_type, cache=True)
print(f'Ready. {len(ancestor_table):,} GO terms, {len(prop_edges):,} hierarchy edges.')

Loading train split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-train-graph.pt ...
Loading val split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-val-graph.pt ...
Loading test split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-test-graph.pt ...
Loaded ProtHGT ESM2 splits  —  proteins: 261,373  GO terms (BP): 27,855

Node types: ['Protein', 'Disease', 'HPO', 'Drug', 'Compound', 'Domain', 'GO_term_P', 'GO_term_F', 'GO_term_C', 'Pathway', 'kegg_Pathway', 'EC_number']
Proteins:   261,373
GO terms:   27,855

Building GO ancestor table (used by RL reward) ...
Building ancestor table for 27,855 GO terms (GO_term_P) ...
Done. Avg ancestors per GO term: 21.4
Ancestor table cached to /tmp/ancestor_table.pkl
Building propagation edge list (used by evaluation) ...
Built 64,409 propagation edge

In [7]:
# Cell 6b: Verify Split Disjointness
# ProtHGT uses a transductive edge-level split: proteins are shared across splits
# (needed for GNN message passing), but the specific annotation edges are held out.
# We check that (protein, GO) pairs don't overlap between train and test.

# Clear .pyc cache so any freshly pulled source files are used
import subprocess, importlib
subprocess.run(["find", "/content/Prot_B_poster", "-name", "*.pyc", "-delete"], capture_output=True)

import src.data.graph_builder as _gb
importlib.reload(_gb)
from src.data.graph_builder import validate_split_disjointness

print("Checking split disjointness...")
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        validate_split_disjointness(train_data, test_data, ttype)
    except (KeyError, ValueError) as e:
        print(f"  {ttype}: {e}")
print("Split validation complete.")


Checking split disjointness...
  GO_term_P: 16,327 proteins shared across splits (transductive edge-level split — expected for ProtHGT)
  GO_term_P: WARNING — 798 (0.8%) (protein, GO) pairs appear in both train and test supervision edges. This is a property of ProtHGT's split; both models see the same overlap so relative comparison remains valid.
  GO_term_P: 797,081 train edges / 101,384 test edges
  GO_term_F: 15,495 proteins shared across splits (transductive edge-level split — expected for ProtHGT)
  GO_term_F: WARNING — 4,179 (4.5%) (protein, GO) pairs appear in both train and test supervision edges. This is a property of ProtHGT's split; both models see the same overlap so relative comparison remains valid.
  GO_term_F: 741,034 train edges / 92,735 test edges
  GO_term_C: 13,186 proteins shared across splits (transductive edge-level split — expected for ProtHGT)
  GO_term_C: WARNING — 5,819 (8.6%) (protein, GO) pairs appear in both train and test supervision edges. This is a prop

In [8]:
# Cell 6c: Pre-compute Information Content Vectors (required for Smin metric)
# IC[t] = -log2(freq_train(t) / N_proteins), computed after label propagation.
# One call per ontology; results stored in ic_vecs dict keyed by ontology shortname.
from src.evaluation.smin import compute_information_content
from src.data.go_hierarchy import build_propagation_edges

ic_vecs = {}
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        pe = build_propagation_edges(train_data, target_type=ttype, cache=True)
        ic = compute_information_content(train_data, ttype, pe)
        ic_vecs[ont] = ic
        print(f"  {ont} ({ttype}): IC ready ({(ic > 0).sum().item()} terms with annotations)")
    except Exception as e:
        print(f"  {ont}: skipped -- {e}")
print("IC computation complete.")


/content/Prot_B_poster/src/evaluation/smin.py:11: SyntaxWarning: invalid escape sequence '\ '
  ru(i,τ) = Σ_{g ∈ true_i \ pred_i(τ)}  IC(g)   — remaining uncertainty


  IC computed: 20,687/27,855 GO terms have annotations
  IC range: [0.62, 18.00]
  bp (GO_term_P): IC ready (20687 terms with annotations)
Built 13,295 propagation edges in topological order (GO_term_F)
  IC computed: 7,133/10,955 GO terms have annotations
  IC range: [0.98, 18.00]
  mf (GO_term_F): IC ready (7133 terms with annotations)
Built 6,127 propagation edges in topological order (GO_term_C)
  IC computed: 2,946/4,075 GO terms have annotations
  IC range: [0.68, 18.00]
  cc (GO_term_C): IC ready (2946 terms with annotations)
IC computation complete.


In [9]:
# ── Cell 7: Initialise Models ──────────────────────────────────────────────────
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.distmult import DistMult
from src.models.reward import RewardModule
from src.utils.logger import TrainingLogger
from src.utils.seed import set_seed

set_seed(cfg.get('seed', 42))

encoder      = CompGCN(train_data, cfg).to(DEVICE)
generator    = Generator(cfg).to(DEVICE)
discriminator = Discriminator(cfg).to(DEVICE)
distmult     = DistMult(hidden_dim=cfg['distmult']['hidden_dim']).to(DEVICE)
reward_module = RewardModule(cfg, distmult, discriminator).to(DEVICE)

total_params = sum(p.numel() for p in encoder.parameters()) + \
               sum(p.numel() for p in generator.parameters()) + \
               sum(p.numel() for p in discriminator.parameters())
print(f'Total trainable parameters: {total_params:,}')

logger = TrainingLogger(
    log_dir='/tmp/runs',
    csv_path=LOG_CSV,
)
print('Logger ready. Run `%load_ext tensorboard` then `%tensorboard --logdir /tmp/runs` to monitor.')

Total trainable parameters: 2,355,201
Logger ready. Run `%load_ext tensorboard` then `%tensorboard --logdir /tmp/runs` to monitor.


In [ ]:
# ── Cell 8: Phase 1 — Pre-training ────────────────────────────────────────────
# SKIP THIS CELL if you already ran Cell 7b above to load an existing checkpoint.
# Trains CompGCN encoder with 4 losses (MSE, cosine, ranking, MMD).
# Generator/Discriminator are frozen during this phase.
# Expected: ~30-40 min on T4 for 50 epochs.

from src.training.pretrain import pretrain

encoder = pretrain(
    encoder=encoder,
    train_data=train_data,
    val_data=val_data,
    cfg=cfg,
    device=DEVICE,
    logger=logger,
)

# Save encoder checkpoint to Drive
import torch, os
ckpt_path = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained.pt')
torch.save(encoder.state_dict(), ckpt_path)
print(f'Saved pretrained encoder → {ckpt_path}')

In [ ]:
# ── Cell 8b: Plot Phase 1 Curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt, os

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
history = logger.history

def plot_metric(ax, key, label, color='steelblue'):
    if key in history:
        steps, vals = zip(*history[key])
        ax.plot(steps, vals, color=color, linewidth=1.5)
        ax.set_title(label); ax.set_xlabel('Step'); ax.grid(alpha=0.3)

for ax, (key, lbl, col) in zip(axes[:3], [
    ('loss/total', 'Total Pretrain Loss', 'steelblue'),
    ('val/cosine_similarity', 'Val Cosine Similarity', 'green'),
    ('embed/protein_norm_mean', 'Protein Emb Norm', 'orange'),
]):
    plot_metric(ax, key, lbl, col)

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'pretrain_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

## Full-Knowledge-Graph Retrain (all entities, no whitelist)

Everything below uses `configs/full_graph.yaml` instead of `default.yaml` — every node
type and edge type in the graph (Drug, Disease, Chembl, everything) is used for message
passing, with no relevance filtering. Requires a Premium GPU runtime (A100, 40GB) —
set this under Runtime > Change runtime type > GPU type before running these cells.

This builds a **separate** encoder (`encoder_full`) from a **separate** config
(`cfg_full`) so it never overwrites the whitelisted `encoder`/`cfg` from Cells 5-8
above — you can compare both runs side by side.

In [10]:
# ── Cell 8f: Load Full-Graph Config ──────────────────────────────────────────
import yaml, os

full_graph_config_path = os.path.join(REPO_DIR, 'configs/full_graph.yaml')
with open(full_graph_config_path) as f:
    cfg_full = yaml.safe_load(f)

cfg_full['data']['drive_root'] = DRIVE_ROOT
CHECKPOINT_DIR_FULL = cfg_full['data']['checkpoint_dir']
os.makedirs(CHECKPOINT_DIR_FULL, exist_ok=True)

whitelist = cfg_full['encoder']['gnn_edge_types']
print('gnn_edge_types:', whitelist if whitelist else '(empty — no filtering, all edges used)')
print('edge_chunk_size:', cfg_full['encoder']['edge_chunk_size'])
print('Checkpoint dir:', CHECKPOINT_DIR_FULL)

gnn_edge_types: (empty — no filtering, all edges used)
edge_chunk_size: 100000
Checkpoint dir: /content/drive/MyDrive/Poster code/checkpoints_full_graph/


In [11]:
# ── Cell 8g: Build CompGCN on the Full Graph ─────────────────────────────────
from src.models.compgcn import CompGCN
from src.utils.seed import set_seed

print('Relations present in train_data:', sorted(set(rel for _, rel, _ in train_data.edge_types)))

set_seed(cfg_full.get('seed', 42))

encoder_full = CompGCN(train_data, cfg_full).to(DEVICE)

n_full_params = sum(p.numel() for p in encoder_full.parameters())
print(f'Full-graph encoder parameters: {n_full_params:,}')
print(f'Node types projected (no whitelist): {list(encoder_full.input_projs.keys())}')

Relations present in train_data: ['Chembl', 'Disease', 'Drug', 'HPO', 'Orthology', 'PPI', 'Pathway', 'domain_function', 'function_function', 'hpodis', 'kegg_dis_drug', 'kegg_dis_path', 'kegg_dis_prot', 'kegg_path_prot', 'protein_domain', 'protein_ec', 'protein_function', 'rev_Chembl', 'rev_Disease', 'rev_Drug', 'rev_HPO', 'rev_Orthology', 'rev_PPI', 'rev_Pathway', 'rev_domain_function', 'rev_function_function', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_kegg_dis_path', 'rev_kegg_dis_prot', 'rev_kegg_path_prot', 'rev_protein_domain', 'rev_protein_ec', 'rev_protein_function']
Full-graph encoder parameters: 1,929,216
Node types projected (no whitelist): ['Protein', 'Disease', 'HPO', 'Drug', 'Compound', 'Domain', 'GO_term_P', 'GO_term_F', 'GO_term_C', 'Pathway', 'kegg_Pathway', 'EC_number']


In [12]:
# ── Cell 8h: Phase 1 on the Full Graph — Pre-training ────────────────────────
# Same pretrain() function, same 4 losses as Cell 8 — only the graph coverage and
# edge_chunk_size differ (see configs/full_graph.yaml). Run a short epoch count
# first (edit cfg_full['pretrain']['epochs']) to confirm it fits in memory and
# val-Fmax isn't collapsed before committing to the full 150-epoch run.

from src.training.pretrain import pretrain
from src.utils.logger import TrainingLogger
import torch, os

logger_full = TrainingLogger(
    log_dir='/tmp/runs_full_graph',
    csv_path=os.path.join(CHECKPOINT_DIR_FULL, 'training_log_full_graph.csv'),
)

encoder_full = pretrain(
    encoder=encoder_full,
    train_data=train_data,
    val_data=val_data,
    cfg=cfg_full,
    device=DEVICE,
    logger=logger_full,
)

ckpt_path_full = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph.pt')
torch.save(encoder_full.state_dict(), ckpt_path_full)
print(f'Saved full-graph pretrained encoder (initial embeddings) → {ckpt_path_full}')

Building annotation index ...
Building false-negative mask lookup ...
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
Starting pre-training: 150 epochs, 797,081 positive pairs
  InfoNCE batch=8,192  temp=0.07  dm_neg=64
  Loss weights: 1.0*InfoNCE(sym+mask)  0.1*cosine  0.5*ComplEx-LP
  Node types the encoder projects (gnn_edge_types whitelist applied):
    Protein: 261,373 nodes × 1280d raw = 1338 MB
    Disease: 5,694 nodes

## Inspect the CompGCN Embeddings Directly

Nothing before this point ever exposes CompGCN's actual output vectors to you --
`pretrain()` and `evaluate_all()` both call `encoder(data)` internally, use the
result immediately, and discard it. This cell calls it once, standalone, and
holds onto the real embeddings so you can look at them directly, plus two
concrete checks that don't depend on the downstream Fmax task at all:

1. **Collapse check** -- are the embeddings degenerate (all nearly identical)?
2. **Graph-structure check** -- do proteins that are real PPI partners end up
   closer together in embedding space than random, unconnected pairs? If
   message passing is doing its job, the answer must be yes -- this has
   nothing to do with GO function prediction, it only tests whether the graph
   part of the Graph Neural Network is actually doing something.

In [13]:
# ── Cell 8h2: Inspect CompGCN Embeddings (3-hop, full-graph encoder) ─────────
import torch
import torch.nn.functional as F
import os

encoder_full.eval()
with torch.no_grad():
    protein_embs, go_embs, rel_embs = encoder_full(train_data)

print('=== Shapes ===')
print(f'Protein embeddings:  {tuple(protein_embs.shape)}')
print(f'GO term embeddings:  {tuple(go_embs.shape)}')
print(f'Relation embeddings: {tuple(rel_embs.shape)}')

p_norms = protein_embs.norm(dim=-1)
g_norms = go_embs.norm(dim=-1)
print('\n=== Collapse check ===')
print(f'Protein norm: mean={p_norms.mean():.4f}  std={p_norms.std():.4f}')
print(f'GO norm:      mean={g_norms.mean():.4f}  std={g_norms.std():.4f}')
if p_norms.mean() < 0.01 or p_norms.std() / (p_norms.mean() + 1e-8) < 0.01:
    print('WARNING: protein embeddings look collapsed (near-zero or near-identical norms).')
else:
    print('OK: protein embedding norms look healthy (non-zero, with real spread).')

print('\n=== Graph-structure check: are PPI partners closer than random pairs? ===')
ppi_key = ('Protein', 'PPI', 'Protein')
if ppi_key in train_data.edge_types:
    ppi_edges = train_data[ppi_key].edge_index
    n_sample  = min(5000, ppi_edges.shape[1])
    sel       = torch.randperm(ppi_edges.shape[1])[:n_sample]
    src, dst  = ppi_edges[0, sel], ppi_edges[1, sel]

    p_norm_vec = F.normalize(protein_embs.float(), dim=-1)
    connected_sim = (p_norm_vec[src] * p_norm_vec[dst]).sum(-1).mean().item()

    n_p = protein_embs.shape[0]
    rand_src = torch.randint(0, n_p, (n_sample,))
    rand_dst = torch.randint(0, n_p, (n_sample,))
    random_sim = (p_norm_vec[rand_src] * p_norm_vec[rand_dst]).sum(-1).mean().item()

    print(f'Connected (real PPI) pairs -- avg cosine similarity: {connected_sim:.4f}')
    print(f'Random, unconnected pairs  -- avg cosine similarity: {random_sim:.4f}')
    if connected_sim > random_sim:
        print('PASS: connected proteins are more similar than random -- graph structure is represented.')
    else:
        print('WARNING: connected proteins are NOT more similar than random -- message passing may not be working.')
else:
    print(f'No {ppi_key} edge type in train_data -- skipping this check.')

# Save the raw embeddings so they can be inspected outside this notebook too.
torch.save(protein_embs.cpu(), os.path.join(CHECKPOINT_DIR_FULL, 'protein_embeddings.pt'))
torch.save(go_embs.cpu(),      os.path.join(CHECKPOINT_DIR_FULL, 'go_embeddings.pt'))
print(f'\nSaved protein_embeddings.pt and go_embeddings.pt -> {CHECKPOINT_DIR_FULL}')

=== Shapes ===
Protein embeddings:  (261373, 256)
GO term embeddings:  (27855, 256)
Relation embeddings: (34, 256)

=== Collapse check ===
Protein norm: mean=16.0183  std=0.0320
GO norm:      mean=16.0298  std=0.0222

=== Graph-structure check: are PPI partners closer than random pairs? ===
Connected (real PPI) pairs -- avg cosine similarity: 0.4014
Random, unconnected pairs  -- avg cosine similarity: 0.1638
PASS: connected proteins are more similar than random -- graph structure is represented.

Saved protein_embeddings.pt and go_embeddings.pt -> /content/drive/MyDrive/Poster code/checkpoints_full_graph/


## Leakage-Aware Fmax Check

`validate_split_disjointness` (Cell 6b) already showed that some exact test
(protein, GO) triples also appear in train — 0.8% for BP, 4.5% for MF, 8.6% for CC.
That tells us the leak *rate*, not whether it actually inflates Fmax. This cell
answers that directly: score the same test set twice — once as-is, once with the
leaked triples removed — and compare. If Fmax barely moves, the overlap isn't doing
meaningful work and the baseline comparison stands; if it drops noticeably, that's a
real result to disclose rather than find out from a reviewer.

Set `EVAL_ENCODER` below to whichever encoder just finished training
(`encoder` for the whitelisted run, `encoder_full` for the full-graph run).

In [14]:
# ── Cell 8i: Leakage-Aware Fmax Check ────────────────────────────────────────
from src.data.graph_builder import find_leaked_pairs
from src.evaluation.metrics import evaluate_all

EVAL_ENCODER = encoder  # ← change to encoder_full to check the full-graph run instead

leaked_pairs = find_leaked_pairs(train_data, test_data, target_type)
print(f'Exact test triples also present in train: {len(leaked_pairs):,}')

print('\n--- Raw Fmax (as normally reported) ---')
raw_results = evaluate_all(
    encoder=EVAL_ENCODER, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
)

print('\n--- Leakage-free Fmax (leaked triples excluded from scoring) ---')
clean_results = evaluate_all(
    encoder=EVAL_ENCODER, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    exclude_pairs=leaked_pairs,
)

delta = raw_results['fmax'] - clean_results['fmax']
print(f"\nRaw Fmax:          {raw_results['fmax']:.4f}")
print(f"Leakage-free Fmax: {clean_results['fmax']:.4f}")
print(f"Delta:             {delta:+.4f}", end=' ')
print('(small — overlap is not meaningfully inflating the score)' if abs(delta) < 0.005
      else '(non-trivial — disclose this explicitly when comparing to any baseline)')

Exact test triples also present in train: 798

--- Raw Fmax (as normally reported) ---
Encoding graph  [mode=encoder] ...
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
Loading propagation edges ...
Computing Fmax+Smin [encoder] streaming 32,675 proteins ...
  Score range: [-1.667, 1.875]  (196 threshold candidates, extra resolution in top 10%)
Computing AUROC / AUPR / MCC (sample of 8,000 proteins) ...
Computing Hit@k / MRR

## Cheap Calibration Test — Full-Graph Phase 1 Encoder

The AUROC (0.887) vs. Fmax (0.165) gap above is the classic "ranks well, thresholds
poorly" pattern `train_calibration_head()` (Phase 4) exists to fix. It trains a small
MLP on the **frozen** `encoder_full` embeddings — cheap, no repeated graph forward
passes — then this cell re-scores Fmax via `mode='calibration'` so you can see how much
of that gap is recoverable before spending Phase 2/3 compute on the full graph.

Starts at 20 epochs for a quick look; raise to the default 50 if val-Fmax is still
climbing at the end of the printed log.

In [15]:
# ── Cell 8j: Quick Calibration Test on the Full-Graph Encoder ────────────────
from src.training.train_calibration import train_calibration_head
from src.evaluation.metrics import evaluate_all

calib_head_full = train_calibration_head(
    encoder=encoder_full,
    train_data=train_data,
    val_data=val_data,
    target_type=target_type,
    cfg=cfg_full,
    device=DEVICE,
    epochs=20,               # quick look first -- raise to 50 (default) if still improving
    checkpoint_dir=CHECKPOINT_DIR_FULL,
)

print('\n--- Fmax via calibration head (full-graph Phase 1 encoder) ---')
calib_results = evaluate_all(
    encoder=encoder_full, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg_full,
    device=DEVICE, mode='calibration', calibration_head=calib_head_full,
    ic_vec=ic_vecs.get(cfg_full['data']['ontology']),
)

print(f"\nComplEx (encoder-mode) Fmax: {raw_results['fmax']:.4f}   (from Cell 8i)")
print(f"Calibrated Fmax:             {calib_results['fmax']:.4f}")
print(f"Uplift from calibration:     {calib_results['fmax'] - raw_results['fmax']:+.4f}")

Caching encoder embeddings (frozen) ...
Training CalibrationHead: 20 epochs, 797,081 positives, neg_ratio=100, embed_dim=256
[CalibHead   1/20] loss=1.3984
[CalibHead   2/20] loss=1.0930
[CalibHead   3/20] loss=1.0117
[CalibHead   4/20] loss=0.9518
[CalibHead   5/20] loss=0.8997  val_Fmax=0.0081  *** NEW BEST
[CalibHead   6/20] loss=0.8663
[CalibHead   7/20] loss=0.8362
[CalibHead   8/20] loss=0.8122
[CalibHead   9/20] loss=0.7973
[CalibHead  10/20] loss=0.7722  val_Fmax=0.0069
[CalibHead  11/20] loss=0.7571
[CalibHead  12/20] loss=0.7394
[CalibHead  13/20] loss=0.7242
[CalibHead  14/20] loss=0.7165
[CalibHead  15/20] loss=0.7051  val_Fmax=0.0097  *** NEW BEST
[CalibHead  16/20] loss=0.6985
[CalibHead  17/20] loss=0.6911
[CalibHead  18/20] loss=0.6810
[CalibHead  19/20] loss=0.6740
[CalibHead  20/20] loss=0.6630  val_Fmax=0.0100  *** NEW BEST

Done. Best val Fmax: 0.0100

--- Fmax via calibration head (full-graph Phase 1 encoder) ---
Encoding graph  [mode=calibration] ...
[rel_idx] ava

## Hand Off to the Full-Graph Encoder

Everything above this point built two separate Phase 1 encoders: the original
whitelisted `encoder` (Cell 8) and the enhanced full-graph `encoder_full`
(Cells 8f-8h, 3 hops, bigger training batches, seeded). Phase 2 onward is
written generically against the names `encoder` and `cfg` -- without this cell,
it would silently keep using the smaller whitelisted encoder and ignore
everything the full-graph run just did.

Run this cell to point the rest of the pipeline (Phase 2, 3, 4) at the
full-graph encoder. Skip it only if you deliberately want to continue with
the smaller whitelisted graph instead.

In [16]:
# ── Cell 8k: Hand Off to Full-Graph Encoder for Phase 2+ ─────────────────────
# Downstream cells (Phase 2, Phase 3, Cell 11d, Cell 12a/12b) all reference the
# PLAIN variables `encoder`, `cfg`, `CHECKPOINT_DIR`, `logger`, and the fixed
# filename 'compgcn_pretrained.pt' -- without reassigning ALL of these together:
#  - Phase 2/3 would save into the whitelisted checkpoint folder while actually
#    training the full-graph (3-layer) encoder, and Cell 11d/12a/12b would then
#    try to load the OLD 2-layer whitelisted checkpoint -- a shape-mismatch crash.
#  - Phase 2/3 would log into the SAME history dict Phase 1's whitelisted run
#    already logged 'val/fmax_bp' etc. into, tangling two different runs' points
#    together on the same step axis in Cell 9b/10b's plots.
import shutil, os

encoder        = encoder_full
cfg            = cfg_full
CHECKPOINT_DIR = CHECKPOINT_DIR_FULL
logger         = logger_full

# Downstream cells expect the Phase 1 checkpoint at the plain filename
# 'compgcn_pretrained.pt' inside CHECKPOINT_DIR -- copy the full-graph checkpoint
# there so those cells work unmodified against whichever encoder was handed off.
src_ckpt = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph.pt')
dst_ckpt = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained.pt')
shutil.copyfile(src_ckpt, dst_ckpt)

print('Phase 2 onward will now train/evaluate/log on the full-graph, 3-hop encoder.')
print(f'Checkpoints will now save/load from: {CHECKPOINT_DIR}')
print(f"cfg['encoder']['num_layers'] = {cfg['encoder']['num_layers']}")

Phase 2 onward will now train/evaluate/log on the full-graph, 3-hop encoder.
Checkpoints will now save/load from: /content/drive/MyDrive/Poster code/checkpoints_full_graph/
cfg['encoder']['num_layers'] = 3


In [ ]:
import os
print(os.listdir(CHECKPOINT_DIR_FULL))

### Validated: the cosine-anchor fix works — Phase 2 uses it by default now

This was checked directly before committing to it, not assumed. Real
(ComplEx-scored) Fmax on `val_data`, same encoder lineage throughout:

| Stage | Real Fmax |
|---|---|
| Phase 1 (full-graph, 3-hop) | 0.3470 |
| Phase 2, pre-fix (85 epochs, no anchor) | 0.4544 |
| Phase 2, post-fix (40 epochs, with anchor) | **0.5496** |

Every ranking/probability metric (AUROC, AUPR, MCC, Hit@k, MRR) improved
monotonically at each stage too — not just Fmax. The anchor fix is now
built into `train_adversarial()`'s own encoder-update rule by default
([src/training/adversarial.py](../src/training/adversarial.py)) — Cell 9
below no longer needs a separate diagnostic pass to benefit from it.

In [20]:
!git -C /content/Prot_B_poster fetch origin
!git -C /content/Prot_B_poster reset --hard origin/main
!git -C /content/Prot_B_poster log -1 --oneline   # should show 1dcc1dd or newer

import importlib
import src.training.adversarial
importlib.reload(src.training.adversarial)
from src.training.adversarial import train_adversarial

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.34 KiB | 1.17 MiB/s, done.
From https://github.com/Drjay806/Prot_B_poster
   1e389cb..1dcc1dd  main       -> origin/main
HEAD is now at 1dcc1dd Merge origin/main (live Colab save of the full 100-epoch Phase 2 run)
1dcc1dd (HEAD -> main, origin/main, origin/HEAD) Merge origin/main (live Colab save of the full 100-epoch Phase 2 run)


In [22]:
import torch, os
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator

encoder = CompGCN(train_data, cfg).to(DEVICE)
encoder.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph.pt'), map_location=DEVICE))
generator = Generator(cfg).to(DEVICE)
discriminator = Discriminator(cfg).to(DEVICE)

print("Reloaded clean Phase 1 encoder + fresh generator/discriminator.")

Reloaded clean Phase 1 encoder + fresh generator/discriminator.


In [ ]:
from src.evaluation.metrics import evaluate_all
check = evaluate_all(
    encoder=encoder, generator=None, distmult=distmult, data=val_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode="encoder", ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)
print(f"Starting point Fmax (should be ~0.3470, NOT ~0.26): {check['fmax']:.4f}")

Encoding graph  [mode=encoder] ...
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
Loading propagation edges ...
Computing Fmax+Smin [encoder] streaming 32,669 proteins ...
  Score range: [-6.842, 10.219]  (191 threshold candidates, extra resolution in top 10%)


In [21]:
cfg['adversarial']['epochs'] = 50   # test run first, not the full 100

encoder, generator, discriminator = train_adversarial(
    encoder=encoder, generator=generator, discriminator=discriminator,
    distmult=distmult, train_data=train_data, val_data=val_data,
    ancestor_table=ancestor_table, cfg=cfg, device=DEVICE,
    logger=logger, checkpoint_dir=CHECKPOINT_DIR,
)

[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
[NegativeSampler] tiers — easy: 9,481  medium: 9,403  hard: 8,971
Starting adversarial training: 50 epochs, 797,081 positive pairs
  WGAN + spectral norm | n_critic=5 | beta1=0.0 beta2=0.9
[Adv Epoch 1/50] W_dist=0.205  C_loss=-0.205  G_loss=-0.647  scores(real=0.15 fake=0.13 hard=-0.23)  DistMult=1.187  acc(real=97% rank=80%)  E_anchor=0.000  
[Adv Epoch 2/50] W_dist=0.204  C_l

KeyboardInterrupt: 

In [17]:
# ── Cell 9: Phase 2 — Adversarial Training ────────────────────────────────────
# GAN training: Critic updated n_critic times per Generator update.
# Expected: ~60-90 min on T4 for 100 epochs (longer on the full graph).
#
# Progress is now tracked with the CORRECT metric (ComplEx-scored, matching
# evaluate_all's official Fmax) instead of the cosine-based check used in Phase 1
# and Phase 3 -- Phase 2's own encoder update has no cosine term, so the old
# cosine-based check could decline for many epochs while the real Fmax was
# actually improving (observed directly on this project: cosine check fell from
# 0.41 to 0.23 over 95 epochs while the real Fmax on that same checkpoint was
# 0.4544, above the Phase 1 baseline of 0.3470).
#
# The best checkpoint by this correct metric is now saved automatically to
# adversarial_best.pt as training progresses -- not just once at the very end.

from src.training.adversarial import train_adversarial

encoder, generator, discriminator = train_adversarial(
    encoder=encoder,
    generator=generator,
    discriminator=discriminator,
    distmult=distmult,
    train_data=train_data,
    val_data=val_data,
    ancestor_table=ancestor_table,
    cfg=cfg,
    device=DEVICE,
    logger=logger,
    checkpoint_dir=CHECKPOINT_DIR,
)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')
torch.save({
    'encoder': encoder.state_dict(),
    'generator': generator.state_dict(),
    'discriminator': discriminator.state_dict(),
}, ckpt_path)
print(f'Saved final-epoch adversarial checkpoint -> {ckpt_path}')
print(f'(The BEST checkpoint by real Fmax, which may be from an earlier epoch,')
print(f' was already saved separately to adversarial_best.pt during training.)')

[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
[NegativeSampler] tiers — easy: 9,481  medium: 9,403  hard: 8,971
Starting adversarial training: 100 epochs, 797,081 positive pairs
  WGAN + spectral norm | n_critic=5 | beta1=0.0 beta2=0.9
[Adv Epoch 1/100] W_dist=0.241  C_loss=-0.241  G_loss=-0.362  scores(real=0.07 fake=0.10 hard=-0.45)  DistMult=0.754  acc(real=69% rank=47%)  E_anchor=0.000  
[Adv Epoch 2/100] W_dist=0.250  

In [19]:
# ── Cell 9b: Plot Phase 2 Curves ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (key, lbl, col) in zip(axes, [
    ('loss/disc_total',    'Discriminator Loss',       'crimson'),
    ('loss/gen',           'Generator Loss',            'steelblue'),
    ('disc/real_acc',      'D Accuracy (Real)',         'green'),
    ('disc/fake_acc',      'D Accuracy (Fake)',         'darkorange'),
    ('reward/distmult_mean', 'DistMult Score (mean)',   'purple'),
    ('val/fmax_bp',        'Val Fmax (BP)',             'black'),
]):
    plot_metric(ax, key, lbl, col)

# Reference line for ProtHGT baseline
if 'val/fmax_bp' in history:
    steps, _ = zip(*history['val/fmax_bp'])
    axes[5].axhline(0.7489, color='red', linestyle='--', label='ProtHGT baseline')
    axes[5].legend()

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'adversarial_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

NameError: name 'plt' is not defined

In [ ]:
# ── Cell 10: Phase 3 — RL Fine-tuning ────────────────────────────────────────
# REINFORCE with GO hierarchy penalty. Semantic reward grows via curriculum.
# Best val Fmax checkpoint is auto-saved to Drive.
# Expected: ~30-45 min on T4 for 50 epochs.

from src.training.rl_trainer import train_rl

encoder, generator = train_rl(
    encoder=encoder,
    generator=generator,
    distmult=distmult,
    reward_module=reward_module,
    train_data=train_data,
    val_data=val_data,
    ancestor_table=ancestor_table,
    cfg=cfg,
    device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger,
)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'rl_final.pt')
torch.save({'encoder': encoder.state_dict(), 'generator': generator.state_dict()}, ckpt_path)
print(f'Saved final RL checkpoint → {ckpt_path}')

In [ ]:
# ── Cell 10b: Plot Phase 3 Curves ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (key, lbl, col) in zip(axes, [
    ('reward/total_mean',       'Mean Reward',              'steelblue'),
    ('reward/hierarchy_penalty','Hierarchy Penalty',         'crimson'),
    ('reward/semantic',         'Semantic Reward',           'green'),
    ('curriculum/w3',           'Semantic Weight w3(t)',     'orange'),
    ('grad/gen_norm',           'Generator Grad Norm',       'purple'),
    ('val/fmax_bp',             'Val Fmax (BP)',             'black'),
]):
    plot_metric(ax, key, lbl, col)

if 'val/fmax_bp' in history:
    axes[5].axhline(0.7489, color='red', linestyle='--', label='ProtHGT baseline')
    axes[5].legend()

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'rl_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

## Final Evaluation — Run These For The Poster

The three cells below produce the numbers and table for the poster:

- **Cell 11d** reloads each phase's own checkpoint independently (Phase 1,
  Phase 1+2, Phase 1+2+3) and scores each one the same way. This is the
  correct phase-vs-phase comparison — unlike Cell 11b further down, which
  compares *scoring modes* on whatever weights happen to be loaded, not phases.
- **Cell 11e** trains two shallow (no-graph) knowledge-graph-embedding
  baselines — plain ComplEx and plain DistMult — directly on the protein-GO
  triples, to show how much the CompGCN's graph structure actually buys you.
- **Cell 12c** merges both into the final table, with Fmax, Smin, AUPR, F1,
  MCC, Hit@10, and MRR for every row.


In [ ]:
# ── Cell 11d: True Phase Comparison (reloads each checkpoint independently) ───
# Fixes the bug in Cell 11b below, where "Baseline (encoder only)" and
# "+ Adversarial (encoder)" both scored the SAME in-memory weights (whatever
# was last trained) instead of actually comparing Phase 1 vs Phase 2 vs Phase 3.
import json, os, torch
from src.evaluation.metrics import evaluate_all, print_ablation_table
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator

for f in ['compgcn_pretrained.pt', 'adversarial_checkpoint.pt', 'rl_final.pt']:
    p = os.path.join(CHECKPOINT_DIR, f)
    print(f, '->', 'OK' if os.path.exists(p) else 'MISSING', p)

phase_results = {}

# --- Phase 1 only ---
enc1 = CompGCN(train_data, cfg).to(DEVICE)
enc1.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained.pt'), map_location=DEVICE))
print("\n=== Phase 1 (pretrain only) ===")
phase_results["Phase 1 (pretrain only)"] = {"bp": evaluate_all(
    encoder=enc1, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, baseline_fmax=0.7489, mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)}

# --- Phase 1 + 2 ---
ckpt2 = torch.load(os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt'), map_location=DEVICE)
enc2 = CompGCN(train_data, cfg).to(DEVICE); enc2.load_state_dict(ckpt2['encoder'])
gen2 = Generator(cfg).to(DEVICE); gen2.load_state_dict(ckpt2['generator'])
disc2 = Discriminator(cfg).to(DEVICE); disc2.load_state_dict(ckpt2['discriminator'])
print("\n=== Phase 1+2 (adversarial) ===")
phase_results["Phase 1+2 (adversarial)"] = {"bp": evaluate_all(
    encoder=enc2, generator=gen2, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, baseline_fmax=0.7489, mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]), discriminator=disc2,
)}

# --- Phase 1 + 2 + 3 ---
ckpt3 = torch.load(os.path.join(CHECKPOINT_DIR, 'rl_final.pt'), map_location=DEVICE)
enc3 = CompGCN(train_data, cfg).to(DEVICE); enc3.load_state_dict(ckpt3['encoder'])
gen3 = Generator(cfg).to(DEVICE); gen3.load_state_dict(ckpt3['generator'])
print("\n=== Phase 1+2+3 (full pipeline) ===")
phase_results["Phase 1+2+3 (full)"] = {"bp": evaluate_all(
    encoder=enc3, generator=gen3, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, baseline_fmax=0.7489, mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)}

print("\n" + "="*70 + "\n  TRUE PHASE COMPARISON (each row = its own checkpoint)\n" + "="*70)
print_ablation_table(phase_results)

with open(os.path.join(CHECKPOINT_DIR, "phase_comparison_results.json"), "w") as f:
    json.dump(phase_results, f, indent=2)
print(f"Saved -> {CHECKPOINT_DIR}/phase_comparison_results.json")


In [ ]:
import torch
p_norm = shallow_complex_encoder.protein_emb.weight.norm().item()
g_norm = shallow_complex_encoder.go_emb.weight.norm().item()
r_norm = shallow_complex_encoder.rel_emb.weight.norm().item()
print(f"protein_emb norm: {p_norm:.4f}")
print(f"go_emb norm:      {g_norm:.4f}")
print(f"rel_emb norm:     {r_norm:.4f}")

In [ ]:
# ── Cell 11e: Shallow KGE Baselines (ComplEx-only / DistMult-only, no graph) ──
# Plain embedding-table models trained directly on protein-GO triples, with no
# graph convolution at all. Isolates how much the CompGCN's message passing
# actually contributes over a standard shallow KGC baseline.
from src.models.distmult import DistMult, TrueDistMult   # DistMult class = ComplEx scoring
from src.models.shallow_kge import train_shallow_kge
from src.evaluation.metrics import evaluate_all

HIDDEN_DIM = cfg["encoder"]["output_dim"]

# --- Shallow ComplEx baseline (no graph) ---
complex_scorer = DistMult(hidden_dim=HIDDEN_DIM)
shallow_complex_encoder = train_shallow_kge(
    train_data, target_type, complex_scorer, HIDDEN_DIM,
    device=DEVICE, epochs=50,
)
print("\n=== Shallow ComplEx (no graph) ===")
results_shallow_complex = evaluate_all(
    encoder=shallow_complex_encoder, generator=None, distmult=complex_scorer,
    data=test_data, ancestor_table=ancestor_table, target_type=target_type,
    cfg=cfg, device=DEVICE, baseline_fmax=0.7489,
)

# --- Shallow DistMult baseline (no graph, symmetric) ---
distmult_scorer = TrueDistMult(hidden_dim=HIDDEN_DIM)
shallow_distmult_encoder = train_shallow_kge(
    train_data, target_type, distmult_scorer, HIDDEN_DIM,
    device=DEVICE, epochs=50,
)
print("\n=== Shallow DistMult (no graph) ===")
results_shallow_distmult = evaluate_all(
    encoder=shallow_distmult_encoder, generator=None, distmult=distmult_scorer,
    data=test_data, ancestor_table=ancestor_table, target_type=target_type,
    cfg=cfg, device=DEVICE, baseline_fmax=0.7489,
)


In [ ]:
import json, os
from src.evaluation.metrics import print_ablation_table

with open(os.path.join(CHECKPOINT_DIR, "phase_comparison_results.json")) as f:
    phase_results = json.load(f)

phase_results["ProtHGT-ESM2 (published)"] = {
    "bp": {"fmax": 0.7489, "smin": -1, "aupr": float("nan"),
           "micro_f1": float("nan"), "mcc": float("nan"),
           "hit@10": float("nan"), "mrr": float("nan")}
}

print_ablation_table(phase_results)

with open(os.path.join(CHECKPOINT_DIR, "final_poster_results.json"), "w") as f:
    json.dump(phase_results, f, indent=2)
print(f"Saved -> {CHECKPOINT_DIR}/final_poster_results.json")


## Calibration Head — Run These After Phase 1 Is Complete

These three cells train a small MLP classification head on top of the frozen
Phase 1 encoder, then evaluate it under the same protocol as everything else.
The calibration head fixes the AUROC-vs-Fmax gap: the encoder ranks correctly
(AUROC 0.97) but ComplEx scores are uncalibrated for thresholding. The head
learns calibrated probabilities directly.


In [ ]:
# ── Cell 12a: Train Calibration Head (Phase 4) ───────────────────────────────
# Trains a small MLP on frozen Phase 1 encoder embeddings using weighted BCE.
# ~30-45 min on T4. Saves best checkpoint to Drive.
# Requires: Phase 1 encoder loaded (either from full retrain or Cell 7b).
import os
from src.models.calibration_head import CalibrationHead
from src.training.train_calibration import train_calibration_head

# Load the current Phase 1 encoder (whichever config `cfg` currently points at).
import torch
from src.models.compgcn import CompGCN

enc_calib = CompGCN(train_data, cfg).to(DEVICE)
enc_calib.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, "compgcn_pretrained.pt"), map_location=DEVICE)
)
print("Phase 1 encoder loaded for calibration training.")

calibration_head = train_calibration_head(
    encoder        = enc_calib,
    train_data     = train_data,
    val_data       = val_data,
    target_type    = target_type,
    cfg            = cfg,
    device         = DEVICE,
    epochs         = 50,
    lr             = 1e-3,
    neg_ratio      = 100,
    batch_size     = 512,
    eval_every     = 5,
    checkpoint_dir = CHECKPOINT_DIR,
)
print("Calibration head training complete.")


In [ ]:
# ── Cell 12b: Evaluate Calibration Head ──────────────────────────────────────
# Evaluates the calibration head through the same evaluate_all pipeline,
# so numbers are directly comparable to Phase 1/2/3 results.
import os, torch
from src.evaluation.metrics import evaluate_all
from src.models.calibration_head import CalibrationHead
from src.models.compgcn import CompGCN

# Load best calibration head checkpoint (if not already in memory)
if "calibration_head" not in dir() or calibration_head is None:
    embed_dim = cfg["encoder"]["output_dim"]
    calibration_head = CalibrationHead(embed_dim=embed_dim).to(DEVICE)
    ckpt_path = os.path.join(CHECKPOINT_DIR, "calibration_head_best.pt")
    calibration_head.load_state_dict(
        {k: v.to(DEVICE) for k, v in torch.load(ckpt_path, map_location=DEVICE).items()}
    )
    print(f"Loaded calibration head from {ckpt_path}")

# Also need Phase 1 encoder for embeddings
enc_for_calib = CompGCN(train_data, cfg).to(DEVICE)
enc_for_calib.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, "compgcn_pretrained.pt"), map_location=DEVICE)
)

print("\n=== Calibration Head (Phase 1 encoder + MLP BCE head) ===")
results_calibration = evaluate_all(
    encoder           = enc_for_calib,
    generator         = None,
    distmult          = distmult,
    data              = test_data,
    ancestor_table    = ancestor_table,
    target_type       = target_type,
    cfg               = cfg,
    device            = DEVICE,
    baseline_fmax     = 0.7489,
    mode              = "calibration",
    ic_vec            = ic_vecs.get(cfg["data"]["ontology"]),
    calibration_head  = calibration_head,
)


In [ ]:
# ── Cell 12c: Final Complete Poster Table ────────────────────────────────────
# Merges all results into one table: Phase 1/2/3, calibration head,
# ProtHGT (our grader), ProtHGT (published).
import json, os
from src.evaluation.metrics import print_ablation_table

# Load Phase comparison (from Cell 11d)
with open(os.path.join(CHECKPOINT_DIR, "phase_comparison_results.json")) as f:
    phase_results = json.load(f)

all_results = {
    **phase_results,
}

# Add calibration head result if available
if "results_calibration" in dir() and results_calibration:
    all_results["Phase 1 + Calibration Head (MLP BCE)"] = {"bp": results_calibration}

# Add ProtHGT under our grader if available
prothgt_grader_path = os.path.join(CHECKPOINT_DIR, "prothgt_grader_results.json")
if os.path.exists(prothgt_grader_path):
    with open(prothgt_grader_path) as f:
        pt_grader = json.load(f)
    all_results["ProtHGT ESM2 (our grader)"] = pt_grader

# Always add published number
all_results["ProtHGT ESM2 (published, different eval)"] = {
    "bp": {"fmax": 0.7489, "smin": -1,
           "aupr": float("nan"), "micro_f1": float("nan"),
           "mcc": float("nan"), "hit@10": float("nan"), "mrr": float("nan")}
}

print_ablation_table(all_results)

with open(os.path.join(CHECKPOINT_DIR, "final_poster_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Saved -> {CHECKPOINT_DIR}/final_poster_results.json")


In [ ]:
# ── Optional: TensorBoard ──────────────────────────────────────────────────────
# Run this cell at any time to open TensorBoard and see live training curves.
%load_ext tensorboard
%tensorboard --logdir /tmp/runs/

In [ ]:
# ── Optional: Resume from Checkpoint ──────────────────────────────────────────
# If Colab disconnects mid-training, use this cell to reload from the last checkpoint.

RESUME_PHASE = 'adversarial'   # 'pretrain' | 'adversarial' | 'rl'
RESUME_PATH = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')  # adjust as needed

ckpt = torch.load(RESUME_PATH, map_location=DEVICE)

if RESUME_PHASE == 'pretrain':
    encoder.load_state_dict(ckpt)
    print('Loaded pretrained encoder.')
elif RESUME_PHASE in ('adversarial', 'rl'):
    encoder.load_state_dict(ckpt['encoder'])
    generator.load_state_dict(ckpt['generator'])
    if 'discriminator' in ckpt:
        discriminator.load_state_dict(ckpt['discriminator'])
    print(f'Loaded {RESUME_PHASE} checkpoint.')

In [ ]:
# ── Optional: time of full graph run
import os, time
path = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph.pt')
mtime = os.path.getmtime(path)
print("Last modified:", time.ctime(mtime)

## Optional: ProtHGT Real-Model Comparison (run separately, at the end)

Moved to the bottom deliberately -- these three cells CANNOT run as part of the
normal top-to-bottom pass above, and don't need to. Everything above this point
(Phases 1-4, all diagnostics, the poster table) is self-contained and complete
without this section.

**Why it's separate:** ProtHGT's released model checkpoint was built with
`torch_geometric==2.2.0`. The rest of this notebook needs a modern PyG version for
CompGCN. The two can't be installed in the same runtime at once -- loading
ProtHGT's checkpoint under a modern PyG fails with parameter-shape mismatches
(`HGTConv` was restructured internally between these versions). This isn't a bug
in our code; it's a real version conflict, confirmed by checking ProtHGT's own
`requirements.txt`.

**How to run this section, when you want ProtHGT's real score for comparison:**
1. Open a **separate, fresh Colab runtime** (Runtime > New runtime, or a throwaway
   notebook) -- not this one.
2. Run **Cell 11c-pre** there to confirm the correct ESM2 checkpoint file is on Drive.
3. Run **Cell 11c-inference** there (its header has the exact `pip install` command
   for the old, matching PyG version). This saves `prothgt_scores_bp.pt` to Drive.
4. Come back to **this** notebook/runtime and run **Cell 11c** below -- it only reads
   that saved score file, so it works fine here with the modern PyG install.

In [ ]:
# ── Cell 11c-pre: Run ProtHGT ESM2 BP Model and Save Scores ──────────────────
# Run this ONCE to generate prothgt_scores_bp.pt on Drive.
# After it completes, Cell 11c below feeds those scores through our grader.
#
# IMPORTANT: the model file must be the ESM2-specific checkpoint, not the default
# one -- ProtHGT ships several protein-embedding variants (APAAC/ESM2/TAPE/ProtT5)
# and only the ESM2 one (Protein input dim 1280) matches our own pipeline and the
# 0.7489 baseline number. On GitHub this lives at:
#   models/alternative_protein_embeddings/esm2/prothgt-esm2-model-biological-process.pt
# NOT models/prothgt-model-biological-process.pt (that one is 768-dim input --
# verified locally by loading both checkpoints and checking lin_dict.Protein.weight.shape).
import os, torch, sys

DRIVE_BASE = "/content/drive/MyDrive/Poster code/prothgt"
MODEL_PATH  = None

# Prefer a path containing 'esm2' explicitly -- fall back to any biological-process
# file only as a last resort, with a loud warning, since that's very likely the
# wrong (non-ESM2) variant.
candidates = []
for root, dirs, files in os.walk(DRIVE_BASE):
    for f in files:
        if "biological-process" in f and f.endswith(".pt"):
            candidates.append(os.path.join(root, f))

esm2_candidates = [c for c in candidates if "esm2" in c.lower()]
if esm2_candidates:
    MODEL_PATH = esm2_candidates[0]
elif candidates:
    MODEL_PATH = candidates[0]
    print(f"WARNING: no ESM2-specific checkpoint found under {DRIVE_BASE} -- "
          f"falling back to {MODEL_PATH}, which is very likely the wrong "
          f"(non-ESM2) protein-embedding variant. Download the real one from "
          f"models/alternative_protein_embeddings/esm2/ in the ProtHGT GitHub repo.")

if MODEL_PATH is None:
    raise FileNotFoundError(
        "No *-biological-process.pt found under "
        f"{DRIVE_BASE}. Download prothgt-esm2-model-biological-process.pt from "
        "https://github.com/HUBioDataLab/ProtHGT/tree/main/models/alternative_protein_embeddings/esm2 "
        "and place it under this Drive folder."
    )
print(f"Using model: {MODEL_PATH}")
print(f"  Size: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")

ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
protein_dim = ckpt['lin_dict.Protein.weight'].shape[1]
print(f"Protein input dim in this checkpoint: {protein_dim}")
assert protein_dim == 1280, (
    f"Expected 1280 (ESM2) but got {protein_dim} -- this is the wrong checkpoint variant. "
    f"Re-check MODEL_PATH above."
)
print("Confirmed: this is the ESM2 variant. Proceed to Cell 11c-inference --")
print("but read its header first: it must run in a SEPARATE Colab runtime, not this one.")

In [ ]:
# -- Cell 11c-inference: Run ProtHGT Inference (SEPARATE runtime required) --
#
# DO NOT run this in your main pipeline notebook/runtime. ProtHGT's checkpoint was
# saved under torch_geometric==2.2.0 (see their requirements.txt). Modern PyG
# (2.4+) restructured HGTConv internally -- separate per-node-type k_lin/q_lin/v_lin
# ModuleDicts became a single combined kqv_lin module. Loading this checkpoint under
# a modern PyG install fails with shape-mismatch errors on kqv_lin/k_lin/q_lin/v_lin --
# verified locally: strict-loading this exact checkpoint under PyG 2.6.1 produces
# hundreds of missing/unexpected keys in exactly that pattern. This is a real,
# version-level incompatibility, not a bug in the reconstruction below.
#
# Safe fix: run ONLY this cell in a fresh Colab runtime (Runtime > New runtime,
# or a throwaway notebook) with ProtHGT's exact original versions installed first:
#
#   !pip install -q torch==1.12.1+cpu torch_geometric==2.2.0 \
#       torch_scatter==2.1.0 torch_sparse==0.6.15 \
#       -f https://download.pytorch.org/whl/cpu/torch_stable.html \
#       -f https://data.pyg.org/whl/torch-1.12.0+cpu.html
#
# Then mount Drive, run this cell to generate and save prothgt_scores_bp.pt, and only
# THEN go back to your main (modern-PyG) runtime and run Cell 11c to grade those saved
# scores -- Cell 11c only loads a plain tensor from disk, so it never touches PyG
# version compatibility at all. This keeps the old/new PyG installs fully isolated.

import os, torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HGTConv
from src.data.graph_builder import build_annotation_matrix

import torch_geometric
assert torch_geometric.__version__.startswith('2.2'), (
    f"torch_geometric=={torch_geometric.__version__} detected -- this cell requires "
    f"2.2.x to match how the checkpoint was saved. Run the pip install command in "
    f"this cell's header, in a FRESH runtime, before continuing."
)

MODEL_PATH = None
for root, dirs, files in os.walk('/content/drive/MyDrive/Poster code/prothgt'):
    for f in files:
        if 'esm2' in f.lower() and 'biological-process' in f and f.endswith('.pt'):
            MODEL_PATH = os.path.join(root, f); break
    if MODEL_PATH: break
print('Model:', MODEL_PATH)

SAVE_PATH = os.path.join(CHECKPOINT_DIR, 'prothgt_scores_bp.pt')
ckpt = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)

# Extract exact metadata AND exact input dims from the checkpoint itself -- no guessing.
node_types = sorted(set(k.split('.')[1] for k in ckpt if k.startswith('lin_dict.')))
in_dims    = {nt: ckpt[f'lin_dict.{nt}.weight'].shape[1] for nt in node_types}
edge_type_strs = sorted(set(k[len('convs.0.a_rel.'):] for k in ckpt
                             if k.startswith('convs.0.a_rel.')))
edge_types = [tuple(s.split('__')) for s in edge_type_strs]
metadata   = (node_types, edge_types)
print('Node types:', len(node_types), '  Edge types:', len(edge_types))
print('Input dims:', in_dims)

class _MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # matches mlp.lins.0/1/2/3 from state dict
        self.lins = nn.ModuleList([
            nn.Linear(256, 128), nn.Linear(128, 64),
            nn.Linear(64,  32),  nn.Linear(32,  1),
        ])
    def forward(self, x):
        for i, lin in enumerate(self.lins):
            x = lin(x)
            if i < len(self.lins) - 1:
                x = F.relu(x)
        return x

class ProtHGTInference(nn.Module):
    def __init__(self, in_dims, metadata, hidden=128, heads=8, layers=2):
        super().__init__()
        self.lin_dict = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([
            HGTConv(hidden, hidden, metadata, heads, group='sum')  # PyG 2.2.0 API
            for _ in range(layers)
        ])
        self.mlp = _MLP()

    def get_embeddings(self, data, device):
        x_dict = {}
        for nt, lin in self.lin_dict.items():
            if hasattr(data[nt], 'x') and data[nt].x is not None:
                x_dict[nt] = lin(data[nt].x.float().to(device)).relu()
        valid_et = set(edge_types)
        ei_dict  = {k: v.to(device) for k, v in data.edge_index_dict.items()
                    if k in valid_et}
        for conv in self.convs:
            x_dict = conv(x_dict, ei_dict)
        return x_dict

model = ProtHGTInference(in_dims, metadata)
missing, unexpected = model.load_state_dict(ckpt, strict=False)
print(f'Loaded. Missing: {len(missing)}  Unexpected: {len(unexpected)}')
if missing or unexpected:
    print('  Sample missing:', missing[:5])
    print('  Sample unexpected:', unexpected[:5])
    print('  If either list is non-empty here (with the correct 2.2.0 install), stop and')
    print('  investigate before trusting the scores -- this should be an exact match.')

model.to(DEVICE).eval()

print('Running ProtHGT forward pass (full graph)...')
with torch.no_grad():
    x_dict = model.get_embeddings(train_data, DEVICE)

protein_embs = x_dict['Protein'].cpu()
go_embs      = x_dict['GO_term_P'].cpu()
print('Protein embs:', protein_embs.shape)
print('GO_term_P embs:', go_embs.shape)

_, _, n_p, n_go = build_annotation_matrix(test_data, 'GO_term_P')
print(f'Scoring {n_p:,} proteins x {n_go:,} GO terms...')

scores = torch.zeros(n_p, n_go)
P, G   = 32, 1024   # protein chunk, GO chunk

with torch.no_grad():
    for i in range(0, n_p, P):
        end = min(i + P, n_p)
        C   = end - i
        p_chunk = protein_embs[i:end].to(DEVICE)
        for j in range(0, n_go, G):
            jend  = min(j + G, n_go)
            Gj    = jend - j
            g_chunk = go_embs[j:jend].to(DEVICE)
            p_exp = p_chunk.unsqueeze(1).expand(-1, Gj, -1).reshape(C * Gj, -1)
            g_exp = g_chunk.unsqueeze(0).expand(C, -1, -1).reshape(C * Gj, -1)
            probs = torch.sigmoid(
                model.mlp(torch.cat([p_exp, g_exp], dim=-1))
            ).squeeze(-1).reshape(C, Gj).cpu()
            scores[i:end, j:jend] = probs
        if i % (P * 100) == 0:
            print(f'  {i}/{n_p} proteins scored')

print(f'Score range: [{scores.min():.4f}, {scores.max():.4f}]')
torch.save(scores, SAVE_PATH)
print('Saved:', SAVE_PATH)
print('Now switch back to your MAIN (modern-PyG) runtime and run Cell 11c to grade these scores.')

In [ ]:
# -- Cell 11c: Evaluate ProtHGT Through Our Grader --
# Requires prothgt_scores_bp.pt saved by Cell 27 first.
# Self-contained: no dependency on all_splits or ablation_results.
import os, json, torch
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from src.data.go_hierarchy import build_propagation_edges
from src.data.graph_builder import build_annotation_matrix
from src.evaluation.metrics import compute_ranking_metrics

SCORE_FILE = os.path.join(CHECKPOINT_DIR, 'prothgt_scores_bp.pt')
if not os.path.exists(SCORE_FILE):
    print('Score file not found:', SCORE_FILE)
    print('Run Cell 27 first.')
else:
    print('Loading', SCORE_FILE, '...')
    prothgt_scores = torch.load(SCORE_FILE).float()
    print('Shape:', prothgt_scores.shape)

    ttype = 'GO_term_P'
    row, col, n_p, n_go = build_annotation_matrix(test_data, ttype)
    true_mat = torch.zeros(n_p, n_go, dtype=torch.float32)
    true_mat[row.cpu(), col.cpu()] = 1.0

    prop_edges = build_propagation_edges(test_data, target_type=ttype, cache=True)
    for child_i, parent_i in prop_edges:
        prothgt_scores[:, parent_i] = torch.max(prothgt_scores[:, parent_i], prothgt_scores[:, child_i])
        true_mat[:, parent_i]       = torch.max(true_mat[:, parent_i],       true_mat[:, child_i])
    true_mat = (true_mat > 0.5).float()

    has_annot = true_mat.any(dim=1)
    sm = prothgt_scores[has_annot]; tm = true_mat[has_annot]
    ct_sum = tm.sum(1).clamp(min=1e-8)
    s_min_v, s_max_v = sm.min().item(), sm.max().item()
    t_steps = cfg.get('evaluation', {}).get('threshold_steps', 100)
    best_f1, best_t = 0.0, s_min_v
    for i in range(t_steps + 1):
        t = s_min_v + i * (s_max_v - s_min_v) / t_steps
        pred = (sm >= t).float()
        tp   = (pred * tm).sum(1)
        prec = (tp / pred.sum(1).clamp(1e-8)).mean().item()
        rec  = (tp / ct_sum).mean().item()
        if prec + rec > 0:
            f1 = 2 * prec * rec / (prec + rec)
            if f1 > best_f1: best_f1, best_t = f1, t

    unique_prots = row.cpu().unique()
    sel = unique_prots[torch.randperm(len(unique_prots))[:min(8000, len(unique_prots))]]
    auc_true   = true_mat[sel]; auc_scores = prothgt_scores[sel]
    col_has_pos  = auc_true.sum(0) > 0
    y_true_flat  = auc_true[:, col_has_pos].numpy().ravel()
    y_score_flat = auc_scores[:, col_has_pos].numpy().ravel()
    auroc = float(roc_auc_score(y_true_flat, y_score_flat)) if y_true_flat.sum() > 0 else 0.0
    rank_m = compute_ranking_metrics(auc_scores, auc_true)
    mcc = float(matthews_corrcoef(tm.numpy().astype(int).ravel(),
                                   (prothgt_scores[has_annot] >= best_t).numpy().astype(int).ravel()))

    res = {'fmax': best_f1, 'smin': -1, 'auroc': auroc, 'micro_f1': 0.0,
           'mcc': mcc, **rank_m}
    print('=== ProtHGT ESM2 (our grader, identical protocol) ===')
    print(f'  Fmax:   {best_f1:.4f}  (published under their eval: 0.7489)')
    print(f'  AUROC:  {auroc:.4f}')
    print(f'  Hit@10: {rank_m.get("hit@10", 0):.4f}')
    print(f'  MRR:    {rank_m.get("mrr", 0):.4f}')
    print(f'  MCC:    {mcc:.4f}')

    with open(os.path.join(CHECKPOINT_DIR, 'prothgt_grader_results.json'), 'w') as _f:
        json.dump({'bp': res}, _f, indent=2)
    print('Saved ->', os.path.join(CHECKPOINT_DIR, 'prothgt_grader_results.json'))
